In [1]:
# CARGAR ENTORNO DE TRABAJO
import sys
from pathlib import Path
import polars as pl
import gc

# notebook/jupyter → proyecto/
PROJECT_ROOT = Path("../..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from scr.python.clase_datalake import ParquetStore

%load_ext autoreload
%autoreload 2

# Variables
datalake_dir = "../../data/rawparquet"
dataoutput = "../../data/interim"

anio_ini = 1995
anio_def = 2023
anio_fin = 2025
meses_list = None

dominios = [
    "euros_taric",
    "euros_taric_pais",
    "euros_pais",
    "kg_taric",
    "kg_taric_pais",
    "kg_pais",
    "euros_sectores",
    "euros_sectores_pais",
]

extra_cols = {
    "euros_taric": {"pais": 0},
    "euros_taric_pais": {},  
    "euros_pais": {"cod_taric": 0, "nivel_taric": 0, "cod_sector_economico":"0", "nivel_sector_economico":0},
    "kg_taric": {"pais": 0},
    "kg_taric_pais": {},
    "kg_pais": {"cod_taric": 0, "nivel_taric": 0},
    "euros_sectores": {"pais": 0},
    "euros_sectores_pais": {"cod_sector_economico":"0", "nivel_sector_economico":0},
}

def align_schema(df: pl.DataFrame, schema: dict[str, pl.DataType]) -> pl.DataFrame:
    # Añadir columnas faltantes
    for col, dtype in schema.items():
        if col not in df.columns:
            df = df.with_columns(pl.lit(None).cast(dtype).alias(col))

    # Forzar tipos
    df = df.with_columns(
        [pl.col(col).cast(dtype) for col, dtype in schema.items()]
    )

    # Orden final
    return df.select(list(schema.keys()))


TARIC_EUROS_SCHEMA = {
    "flujo": pl.Int32,
    "anio": pl.Int32,
    "mes": pl.Int32,
    "estado": pl.Int32,
    "pais": pl.Int32,
    "nivel_taric": pl.Int32,
    "cod_taric": pl.Int64,
    "euros": pl.Float64,
}


SECTORES_EUROS_SCHEMA = {
    "flujo": pl.Int32,
    "anio": pl.Int32,
    "mes": pl.Int32,
    "estado": pl.Int32,
    "pais": pl.Int32,
    "nivel_sector_economico": pl.Int32,
    "cod_sector_economico": pl.Utf8,
    "euros": pl.Float64,
}

TARIC_KG_SCHEMA = {
    "flujo": pl.Int32,
    "anio": pl.Int32,
    "mes": pl.Int32,
    "estado": pl.Int32,
    "pais": pl.Int32,
    "nivel_taric": pl.Int32,
    "cod_taric": pl.Int64,   
    "kilogramos": pl.Float64,
}

In [2]:
# Guardaremos los DataFrames en un diccionario por dominio y ámbito
dfs = {}

with ParquetStore(datalake_dir) as store:

    for ambito in ["espana", "madrid"]:
        for dominio in dominios:
            print(f"\n📍 Procesando {ambito}/{dominio}")

            # Merge en memoria
            lf = store.merge_parquet_range(
                ambito=ambito,
                dominio=dominio,
                anio_inicio=anio_ini,
                anio_definitivo=anio_def,
                anio_fin=anio_fin,
                meses=meses_list,
                lazy=True
            )

            # Convertir a DataFrame (collect) para trabajar en memoria
            df = lf.collect()

            # Añadir columnas extra si existen
            if dominio in extra_cols and extra_cols[dominio]:
                for col_name, value in extra_cols[dominio].items():
                    df = df.with_columns(pl.lit(value).alias(col_name))

            # Guardamos en el diccionario
            dfs[(ambito, dominio)] = df

            print(f"   ✓ {ambito}/{dominio} listo con shape {df.shape}")
            
            # Liberar memoria del lazyframe
            del lf
            gc.collect()
    
        print("\n📍 Procesando totalesccaa")

    lf_ccaa = store.merge_parquet_range(
        ambito="totalesccaa",
        dominio="",               
        anio_inicio=anio_ini,
        anio_definitivo=anio_def,
        anio_fin=anio_fin,
        meses=meses_list,
        lazy=True
    )

    df_ccaa = (
        lf_ccaa
        .filter(pl.col("cod_comunidad").is_in([15, 99]))
        .collect()
    )

    dfs[("totalesccaa", "total")] = df_ccaa
    print(f"   ✓ totalesccaa filtrado listo con shape {df_ccaa.shape}")

    del lf_ccaa
    gc.collect()

2026-01-09 16:25:44 - INFO - Agregando espana/euros_taric 1995-2025 (def hasta 2023)
2026-01-09 16:25:44 - INFO - ✓ 370 parquets añadidos a la agregación



📍 Procesando espana/euros_taric


2026-01-09 16:25:44 - INFO - Agregando espana/euros_taric_pais 1995-2025 (def hasta 2023)
2026-01-09 16:25:44 - INFO - ✓ 370 parquets añadidos a la agregación


   ✓ espana/euros_taric listo con shape (18851314, 8)

📍 Procesando espana/euros_taric_pais


2026-01-09 16:25:45 - INFO - Agregando espana/euros_pais 1995-2025 (def hasta 2023)
2026-01-09 16:25:45 - INFO - ✓ 370 parquets añadidos a la agregación
2026-01-09 16:25:45 - INFO - Agregando espana/kg_taric 1995-2025 (def hasta 2023)
2026-01-09 16:25:45 - INFO - ✓ 370 parquets añadidos a la agregación


...

   ✓ madrid/kg_taric_pais listo con shape (82836909, 8)

📍 Procesando madrid/kg_pais
   ✓ madrid/kg_pais listo con shape (124811, 8)

📍 Procesando madrid/euros_sectores


2026-01-09 16:25:51 - INFO - Agregando madrid/euros_sectores_pais 1995-2025 (def hasta 2023)
2026-01-09 16:25:51 - INFO - ✓ 370 parquets añadidos a la agregación


   ✓ madrid/euros_sectores listo con shape (140098, 8)

📍 Procesando madrid/euros_sectores_pais


2026-01-09 16:25:51 - INFO - Agregando totalesccaa/ 1995-2025 (def hasta 2023)
2026-01-09 16:25:51 - INFO - ✓ 370 parquets añadidos a la agregación


   ✓ madrid/euros_sectores_pais listo con shape (7167985, 8)

📍 Procesando totalesccaa
   ✓ totalesccaa filtrado listo con shape (1480, 7)


In [3]:
final_dfs = {}

# Filtrar totales de CCAA
df_totales = dfs.pop(("totalesccaa", "total"), None)

if df_totales is None:
    raise ValueError("No se encontró el DataFrame de totalesccaa")

for ambito, cod_comunidad in [("espana", 99), ("madrid", 15)]:

    # ---------------- TARIC - EUROS ----------------
    total_taric_euros = df_totales.filter(pl.col("cod_comunidad") == cod_comunidad).with_columns(
        pl.lit(0).alias("cod_taric"),
        pl.lit(0).alias("nivel_taric"),
        pl.lit(0).alias("pais"),
    )
    total_taric_euros = align_schema(total_taric_euros, TARIC_EUROS_SCHEMA)

    dfs_list = [
        dfs.pop((ambito, "euros_taric"), None),
        dfs.pop((ambito, "euros_taric_pais"), None),
        dfs.pop((ambito, "euros_pais"), None),
        total_taric_euros,
    ]
    dfs_list = [df for df in dfs_list if df is not None]

    final_dfs[f"{ambito}_euros_taric"] = pl.concat([align_schema(df, TARIC_EUROS_SCHEMA) for df in dfs_list], how="vertical")
    
    # Liberar memoria
    del dfs_list, total_taric_euros
    gc.collect()

    # ---------------- TARIC - KG ----------------
    total_taric_kg = df_totales.filter(pl.col("cod_comunidad") == cod_comunidad).with_columns(
        pl.lit(0).alias("cod_taric"),
        pl.lit(0).alias("nivel_taric"),
        pl.lit(0).alias("pais"),
    )
    total_taric_kg = align_schema(total_taric_kg, TARIC_KG_SCHEMA)

    dfs_list = [
        dfs.pop((ambito, "kg_taric"), None),
        dfs.pop((ambito, "kg_taric_pais"), None),
        dfs.pop((ambito, "kg_pais"), None),
        total_taric_kg,
    ]
    dfs_list = [df for df in dfs_list if df is not None]

    final_dfs[f"{ambito}_kg_taric"] = pl.concat([align_schema(df, TARIC_KG_SCHEMA) for df in dfs_list], how="vertical")

    del dfs_list, total_taric_kg
    gc.collect()

    # ---------------- SECTORES - EUROS ----------------
    total_sectores = df_totales.filter(pl.col("cod_comunidad") == cod_comunidad).with_columns(
        pl.lit("0").alias("cod_sector_economico"),
        pl.lit(0).alias("nivel_sector_economico"),
        pl.lit(0).alias("pais"),
    )
    total_sectores = align_schema(total_sectores, SECTORES_EUROS_SCHEMA)

    dfs_list = [
        dfs.pop((ambito, "euros_sectores"), None),
        dfs.pop((ambito, "euros_sectores_pais"), None),
        dfs.pop((ambito, "euros_pais"), None),
        total_sectores,
    ]
    dfs_list = [df for df in dfs_list if df is not None]

    final_dfs[f"{ambito}_euros_sectores"] = pl.concat([align_schema(df, SECTORES_EUROS_SCHEMA) for df in dfs_list], how="vertical")

    del dfs_list, total_sectores
    gc.collect()

    print(f"✅ Finalizados {ambito} con totales")

# Liberamos el resto de dfs que ya no necesitamos
del dfs, df_totales
gc.collect()

# ---------------- Guardar y crear variables accesibles ----------------
for name, df in final_dfs.items():
    # Renombrar columna 'anio' a 'año' si existe
    if 'anio' in df.columns:
        df = df.rename({'anio': 'año'})

    # Construir nombre con años
    # name_with_years = f"{name}_{anio_ini}_{anio_def}_{anio_fin}"
    
    # Guardar en parquet
    # output_path = Path(dataoutput) / f"{name_with_years}.parquet"
    output_path = Path(dataoutput) / f"{name}.parquet"
    df.write_parquet(output_path)
    
    # Crear variable global para exploración
    # globals()[name_with_years] = df
    globals()[name] = df

print("💾 Todos los DataFrames guardados y accesibles como variables.")


✅ Finalizados espana con totales
✅ Finalizados madrid con totales
💾 Todos los DataFrames guardados y accesibles como variables.
